# Get results out of my WandB automatically

In [1]:
import wandb
import numpy as np
from itertools import product
import pandas as pd

api = wandb.Api()
runs = api.runs("labeebah-islaam/world_models")

In [ ]:
modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]
metric = "actor_critic/eval/planned_cumulative_reward"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

# Filter runs that do NOT start with 'pruned'
runs = [r for r in runs if not r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched = []
    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner)
            ):
                val = get_last_value(run)
                if val is not None:
                    matched.append(val)
        except KeyError:
            continue

    if len(matched) >= 3:
        print(f"Matched {len(matched)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        arr = np.array(matched[:3])  # only take first 3
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": arr.mean(),
            "std": arr.std()
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": None,
            "std": None
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("sweep_results.csv", index=False)

       mode  planning_steps  threshold  inner_steps  mean   std
0    reward               0        2.0            0  None  None
1    reward               0        2.0            1  None  None
2    reward               0        2.0            2  None  None
3    reward               0        2.0            5  None  None
4    reward               0        1.5            0  None  None
..      ...             ...        ...          ...   ...   ...
163   value              20        1.5            5  None  None
164   value              20        1.0            0  None  None
165   value              20        1.0            1  None  None
166   value              20        1.0            2  None  None
167   value              20        1.0            5  None  None

[168 rows x 6 columns]


In [2]:
import numpy as np
import pandas as pd
from itertools import product

modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]

metric = "actor_critic/eval/planned_cumulative_reward"
meta_metric = "meta_planning_depth"
eval_step_key = "eval_step"
num_planning_steps_key = "actor_critic/eval/num_planning_steps"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

def get_last_eval_step(run):
    """Episode length = last eval_step for the run."""
    try:
        hist = run.history(keys=[eval_step_key], pandas=True)
        if not hist.empty:
            vals = hist[eval_step_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_num_planning_steps(run):
    """Single scalar; prefer summary, fall back to last history value."""
    try:
        v = run.summary.get(num_planning_steps_key, None)
        if v is not None:
            return float(v)
    except Exception:
        pass
    try:
        hist = run.history(keys=[num_planning_steps_key], pandas=True)
        if not hist.empty:
            vals = hist[num_planning_steps_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_meta_depth_counts(run):
    """Counts of meta_planning_depth occurrences per run for depths 1..4."""
    try:
        hist = run.history(keys=[meta_metric], pandas=True)
        if not hist.empty:
            vals = hist[meta_metric].dropna().astype(int).values
            return {d: int(np.sum(vals == d)) for d in [1, 2, 3, 4]}
    except Exception:
        pass
    return {d: 0 for d in [1, 2, 3, 4]}

# Filter runs that DO start with 'pruned'
runs = [r for r in runs if r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched_records = []

    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            # enforce planning_depth condition
            required_depth = 3 if (step == 0 or inner == 0) else 5

            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner) and
                int(cfg["evaluation"]["planning_depth"]) == required_depth
            ):
                rec = {}
                rec["reward"] = get_last_value(run)
                rec["ep_len"] = get_last_eval_step(run)
                rec["meta_counts"] = get_meta_depth_counts(run)
                rec["num_planning_steps"] = get_num_planning_steps(run)

                # require reward & episode length to include the run
                if rec["reward"] is not None and rec["ep_len"] is not None:
                    matched_records.append(rec)
        except KeyError:
            continue

    if len(matched_records) >= 3:
        print(f"Matched {len(matched_records)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        recs = matched_records[:3]

        arr_vals = np.array([r["reward"] for r in recs], dtype=float)
        arr_lens = np.array([r["ep_len"] for r in recs], dtype=float)

        # meta-depth counts per run
        counts_per_run = {d: np.array([r["meta_counts"][d] for r in recs], dtype=float) for d in [1, 2, 3, 4]}
        counts_mean = {d: float(np.mean(counts_per_run[d])) for d in [1, 2, 3, 4]}
        counts_std  = {d: float(np.std(counts_per_run[d]))  for d in [1, 2, 3, 4]}

        # num_planning_steps per run (allow NaN)
        nps = np.array([r["num_planning_steps"] if r["num_planning_steps"] is not None else np.nan for r in recs], dtype=float)
        nps_mean = float(np.nanmean(nps)) if np.any(~np.isnan(nps)) else None
        nps_std  = float(np.nanstd(nps))  if np.any(~np.isnan(nps)) else None

        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": arr_vals.mean(),
            "reward_std": arr_vals.std(),
            "ep_length_mean": arr_lens.mean(),
            "ep_length_std": arr_lens.std(),
            "num_planning_steps_mean": nps_mean,
            "num_planning_steps_std": nps_std,
            **{f"meta_depth_{d}_mean": counts_mean[d] for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": counts_std[d] for d in [1, 2, 3, 4]},
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": None,
            "reward_std": None,
            "ep_length_mean": None,
            "ep_length_std": None,
            "num_planning_steps_mean": None,
            "num_planning_steps_std": None,
            **{f"meta_depth_{d}_mean": None for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": None for d in [1, 2, 3, 4]},
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("pruned_sweep_results.csv", index=False)

Matched 3 runs for mode=reward, steps=0, thres=2, inner=0
Matched 3 runs for mode=reward, steps=1, thres=2, inner=0
Matched 3 runs for mode=reward, steps=1, thres=2, inner=1
Matched 3 runs for mode=reward, steps=1, thres=2, inner=2
Matched 3 runs for mode=reward, steps=1, thres=2, inner=5
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=0
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=1
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=2
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=5
Matched 3 runs for mode=reward, steps=1, thres=1, inner=0
Matched 3 runs for mode=reward, steps=1, thres=1, inner=1
Matched 3 runs for mode=reward, steps=1, thres=1, inner=2
Matched 3 runs for mode=reward, steps=1, thres=1, inner=5
Matched 3 runs for mode=reward, steps=2, thres=2, inner=0
Matched 3 runs for mode=reward, steps=2, thres=2, inner=1
Matched 3 runs for mode=reward, steps=2, thres=2, inner=2
Matched 3 runs for mode=reward, steps=2, thres=2, inner=5
Matche